In [1]:
import streamlit as st
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

st.set_page_config(page_title="Customer Segmentation RFM", layout="wide")
st.title("Customer Segmentation - RFM Analysis")
st.caption("Segment customers into Champions, Loyal, At Risk, Lost")

uploaded_file = st.file_uploader("Upload sales CSV (date, customer_id, sales)", type=["csv"])

def generate_sample():
    rng = np.random.default_rng(42)
    customers = [f"CUST_{i:04d}" for i in range(1, 501)]
    rows = []
    for cust in customers:
        orders = rng.integers(1, 15)
        for _ in range(orders):
            date = datetime.now() - timedelta(days=int(rng.integers(1, 400)))
            sales = float(rng.integers(20, 1000))
            rows.append({"date": date, "customer_id": cust, "sales": sales})
    return pd.DataFrame(rows)

if uploaded_file:
    df = pd.read_csv(uploaded_file)
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
else:
    st.info("No file uploaded - using sample data (500 customers)")
    df = generate_sample()

# RFM Calculation
snapshot_date = df['date'].max() + timedelta(days=1)
rfm = df.groupby('customer_id').agg(
    recency=('date', lambda x: (snapshot_date - x.max()).days),
    frequency=('date', 'count'),
    monetary=('sales', 'sum')
).reset_index()

# Scoring 1-5
rfm['r_score'] = pd.qcut(rfm['recency'], 5, labels=[5,4,3,2,1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'], 5, labels=[1,2,3,4,5]).astype(int)
rfm['rfm_score'] = rfm['r_score']*100 + rfm['f_score']*10 + rfm['m_score']

def segment(row):
    if row['r_score'] >=4 and row['f_score'] >=4: return "Champions"
    elif row['r_score'] >=3 and row['f_score'] >=3: return "Loyal"
    elif row['r_score'] <=2 and row['f_score'] >=3: return "At Risk"
    else: return "Lost / Hibernating"
rfm['segment'] = rfm.apply(segment, axis=1)

# KPIs
c1,c2,c3,c4 = st.columns(4)
c1.metric("Total Customers", f"{len(rfm)}")
c2.metric("Champions", f"{len(rfm[rfm['segment']=='Champions'])}")
c3.metric("Avg Monetary", f"${rfm['monetary'].mean():,.0f}")
c4.metric("At Risk", f"{len(rfm[rfm['segment']=='At Risk'])}")

# Charts
left,right = st.columns(2)
with left:
    fig, ax = plt.subplots(figsize=(8,4))
    sns.countplot(data=rfm, x='segment', palette='viridis', ax=ax)
    ax.set_title("Customers by Segment")
    plt.xticks(rotation=20)
    st.pyplot(fig)
with right:
    fig, ax = plt.subplots(figsize=(8,4))
    sns.scatterplot(data=rfm, x='recency', y='monetary', hue='segment', palette='viridis', ax=ax)
    ax.set_title("Recency vs Monetary")
    st.pyplot(fig)

st.dataframe(rfm.sort_values('monetary', ascending=False).head(10), use_container_width=True)
st.download_button("Download RFM", rfm.to_csv(index=False), "rfm_segments.csv", "text/csv")

2026-10-06 22:10:31.158 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-06 22:10:31.162 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-06 22:10:32.596 
  command:

    streamlit run C:\Users\zubair.hanif\AppData\Roaming\Python\Python313\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-10-06 22:10:32.596 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-06 22:10:32.597 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-06 22:10:32.598 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-06 22:10:32.598 Thread 'MainThread': missing ScriptRunContext! This warnin

False